# Notebook 1 – Import

Loads the IHFC heat-flow database, applies a multi-step quality-filter pipeline,
builds the `ref` Grid and imports observables, computes tectonic-province-based
sample weights, plots the KDE diagnostic, and builds three grids.

**Sections**
1. Imports and configuration
2. Helper functions
3. Load IHFC database and filtering pipeline
4. Build `ref` Grid and import observables
5. Tectonic-province-based sample weights
6. KDE diagnostic plot (unweighted steps + final weighted curve)
7. Build Antarctica 5 km prediction grid
8. Build Greenland 5 km prediction grid
9. Summary validation
10. Quick-look: `ref` heat-flow map (`q` only)
11. Quick-look: all observables – ref, Antarctica, Greenland

**Outputs**
- `data/IHFC_obs.parquet`  — reference database with observables & weights
- `data/antarctica.parquet`
- `data/greenland.parquet`
- `fig/IHFC_q_distribution_cleaning_steps.png`
- `data/IHFC_distribution_stats.csv`


## 1. Imports and configuration

In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import sys, warnings, logging
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from pyproj import Transformer
from sklearn.cluster import DBSCAN
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import seaborn as sns
import cmcrameri.cm as cmc
import importlib

from lib.agrid import Grid
import config; importlib.reload(config); from config import *
import recipe; importlib.reload(recipe)

print("All imports successful")


AssertionError: obs_sweep has features not in recipe!

## 2. Helper functions

In [ ]:
def _custom_mode(x):
    mode = x.mode()
    return mode.iloc[0] if len(mode) > 0 else x.iloc[0]


def mask_target(df, polygon_file=local_data / "land_5dg_buffer.gpkg"):
    """Boolean mask – True for points INSIDE Antarctica/Greenland 5° buffer."""
    land = gpd.read_file(polygon_file)
    gdf_points = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326"
    )
    if land.crs != gdf_points.crs:
        land = land.to_crs(gdf_points.crs)
    joined = gpd.sjoin(gdf_points, land[["geometry"]], how="left", predicate="within")
    return joined.index_right.notna()


def record_step(steps, data, label):
    """Append a distribution snapshot to steps list for later KDE plotting."""
    steps.append({
        "label":  label,
        "q":      data.copy(),
        "n":      len(data),
        "mean":   float(np.mean(data)),
        "median": float(np.median(data)),
    })


def weighted_quantile(values, quantiles, sample_weight=None):
    """Weighted percentile (Stål et al. 2022)."""
    values = np.asarray(values)
    quantiles = np.atleast_1d(quantiles)
    if sample_weight is None:
        sample_weight = np.ones_like(values, dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]
    cumulative = np.cumsum(sample_weight)
    cumulative = (cumulative - 1) / (cumulative[-1] - 1)
    result = np.interp(quantiles, cumulative, values)
    return float(result[0]) if result.shape == (1,) else result


def compute_kde(xeval, data, bw_scale=0.1, weights=None):
    kde = gaussian_kde(data, weights=weights)
    kde.set_bandwidth(bw_method=bw_scale * kde.factor)
    y_density = kde(xeval)
    n_eff = len(data) if weights is None else np.sum(weights)
    y_counts = y_density * n_eff
    return kde, y_counts, n_eff


def calculate_weights(
    df,
    polygon_file=data_root / 'global_tectonics-main/plates&provinces/global_gprv_wage.shp',
    w_min=0.1,
    w_max=10.0,
    attribute='prov_type',
):
    """
    Tectonic-province-based sample weights (Stål et al., 2022).
    Weights are proportional to province area / count, normalised so
    sum(weights) == len(df), then clipped to [w_min, w_max].
    """
    gprv = gpd.read_file(polygon_file)
    if gprv.crs is None:
        gprv = gprv.set_crs('EPSG:4326')
    elif gprv.crs.to_string() != 'EPSG:4326':
        gprv = gprv.to_crs('EPSG:4326')

    gprv_eq = gprv.to_crs('ESRI:54009')
    gprv_eq['area_m2'] = gprv_eq.geometry.area
    area_by_type = gprv_eq.groupby(attribute)['area_m2'].sum()

    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['lon'], df['lat']),
        crs='EPSG:4326',
    )
    gdf['orig_index'] = gdf.index

    joined = gpd.sjoin(
        gdf, gprv[['EPSG:4326'][0:0] + [attribute, 'geometry']].to_crs('EPSG:4326'),
        how='left', predicate='intersects',
    )
    prov_col = [c for c in joined.columns if attribute in c][0]

    prov_by_point = (
        joined[['orig_index', prov_col]]
        .dropna(subset=[prov_col])
        .groupby('orig_index')[prov_col].first()
    )
    gdf[attribute] = np.nan
    gdf.loc[prov_by_point.index, attribute] = prov_by_point.values

    n_no_poly = gdf[attribute].isna().sum()
    if n_no_poly > 0:
        print(f'Warning: {n_no_poly} points outside all {attribute} polygons → weight=1')

    counts_by_type = gdf.loc[gdf[attribute].notna(), attribute].value_counts()
    common_types = counts_by_type.index.intersection(area_by_type.index)

    A_tot = area_by_type.loc[common_types].sum()
    N_valid = counts_by_type.loc[common_types].sum()
    ideal_counts = N_valid * area_by_type.loc[common_types] / A_tot
    weights_by_type = (ideal_counts / counts_by_type.loc[common_types]).to_dict()

    N_total = len(gdf)
    weights = np.ones(N_total, dtype=float)
    for t, w_t in weights_by_type.items():
        weights[(gdf[attribute] == t).to_numpy()] = w_t

    # normalise → sum == N_total, then clip
    weights = weights / weights.sum() * N_total
    weights = np.clip(weights, w_min, w_max)
    weights = weights / weights.sum() * N_total  # renormalise after clip

    print('\nWeighting summary by prov_type:')
    for t in common_types:
        n_t = counts_by_type.loc[t]
        A_km2 = area_by_type.loc[t] / 1e6
        mw = float(weights[(gdf[attribute] == t).to_numpy()].mean())
        print(f'  {t}: n={n_t:6d}, area={A_km2:10.1f} km², mean weight={mw:6.3f}')
    print(f'Total samples: {N_total}, sum(weights): {weights.sum():.3f}')
    return weights

## 3. Load IHFC database and apply filtering pipeline

Each step records a distribution snapshot in `steps`.
Steps: (1) basic NaN drop, (2) remove target-region rows, (3) remove deep-ocean rows,
(4) remove low-quality codes U4/M4, (5) DBSCAN spatial aggregation.


In [ ]:
ref_df_all = pd.read_excel(GHFDB_file, skiprows=5, nrows=3)
print(ref_df_all.columns.tolist())


In [ ]:
force_read = True  # True → always read from Excel; False → use parquet cache

if parquet_ref.exists() and not force_read:
    print(f'✓ Loading cached IHFC data from {parquet_ref}')
    ref_df = pd.read_parquet(parquet_ref)
    steps = []
    print(f'  Loaded {len(ref_df):,} locations')
else:
    print('Reading IHFC Global Heat Flow Database …')

    ref_df = (
        pd.read_excel(GHFDB_file, skiprows=5, usecols='A,D,E,F,BR,BT')
        .rename(columns={'lat_NS': 'lat', 'long_EW': 'lon'})
    )
    for col in ['q', 'lat', 'lon', 'elevation']:
        ref_df[col] = pd.to_numeric(ref_df[col], errors='coerce')

    # mW/m² → W/m²
    print(f'\n✓ Converting q: {ref_df["q"].min():.1f}–{ref_df["q"].max():.1f} mW/m²')
    ref_df['q'] *= milli
    print(f'  After: {ref_df["q"].min():.6f}–{ref_df["q"].max():.6f} W/m²')

    # Hard clip
    n_lo = (ref_df['q'] < q_clip_min).sum()
    n_hi = (ref_df['q'] > q_clip_max).sum()
    ref_df['q'] = ref_df['q'].clip(q_clip_min, q_clip_max)
    print(f'\n✓ Clipped {n_lo:,} below {q_clip_min} and {n_hi:,} above {q_clip_max} W/m²')

    # Clean categorical columns
    ref_df['Domain'] = ref_df['Domain'].map(
        lambda x: str(x).strip()[0].upper() if pd.notna(x) else np.nan)
    ref_df['Quality_Code'] = ref_df['Quality_Code'].map(
        lambda x: str(x).replace(' ', '')[:13] if pd.notna(x) else np.nan)

    steps = []
    ref_df = ref_df.dropna(subset=['q', 'lat', 'lon'])
    print(f'\nAfter dropna: {len(ref_df):,} rows')
    record_step(steps, ref_df['q'].values, 'After basic cleaning')

    # Remove Antarctica + Greenland (5° buffer)
    n_before = len(ref_df)
    ref_df = ref_df[~mask_target(ref_df)]
    print(f'Dropped {n_before - len(ref_df):,} rows in target regions')
    record_step(steps, ref_df['q'].values, 'After target removal')

    # Deep-ocean filter
    n_before = len(ref_df)
    ref_df = ref_df[ref_df['elevation'] >= deep_ocean_threshold]
    print(f'Dropped {n_before - len(ref_df):,} deep-ocean rows (< −5000 m)')
    record_step(steps, ref_df['q'].values, 'After deep-ocean filter')

    # Quality-code filter
    n_before = len(ref_df)
    qc = ref_df['Quality_Code'].fillna('')
    ref_df = ref_df[~(qc.str.startswith('U4.') | qc.str.contains('.M4.', regex=False))]
    print(f'Dropped {n_before - len(ref_df):,} low-quality rows (U4/M4)')
    record_step(steps, ref_df['q'].values, 'After quality filter')

    # DBSCAN spatial aggregation
    coords_rad = np.radians(ref_df[['lat', 'lon']].values)
    clust = DBSCAN(eps=ref_clustering_radius_km / 6_371.0, min_samples=1,
                   metric='haversine').fit(coords_rad)
    ref_df['cluster'] = clust.labels_

    n_before = len(ref_df)
    ref_df = (
        ref_df.groupby('cluster', as_index=False)
        .agg(lat=('lat', 'mean'), lon=('lon', 'mean'),
             q=('q', 'mean'), elevation=('elevation', 'mean'),
             Domain=('Domain', _custom_mode),
             Quality_Code=('Quality_Code', _custom_mode))
    )
    print(f'\n✓ Aggregated {n_before - len(ref_df):,} clustered locations')
    print(f'  Final database: {len(ref_df):,} unique locations')
    record_step(steps, ref_df['q'].values, 'After DBSCAN aggregation')

    ref_df.to_parquet(parquet_ref)
    print(f'  Cached to {parquet_ref}')

print(f'\nHeat flow statistics (W/m²):')
print(ref_df['q'].describe())
print(f'Columns: {list(ref_df.columns)}')

## 4. Build `ref` Grid and import observables

In [ ]:
# ── Reference grid ──────────────────────────────────────────────────────────
ref = Grid(
    lats=ref_df.lat.values, lons=ref_df.lon.values,
    name="IHFC", crs=4326, verbose=False,
    log_file=str(log_dir / "read_ref.log"),
)
for col in ["q", "elevation", "Domain", "Quality_Code"]:
    if col in ref_df.columns:
        ref.df[col] = ref_df[col].values

dd = ref.organise_recipe(recipe.dd, verbose=True)

imported_ref = ref.read_recipe(dd, verbose=True)
print(f"Ref: {len(imported_ref)} observables imported")
print(f"Columns: {list(ref.df.columns)}")

## 5. Tectonic-province-based sample weights

Weights are computed on `ref.df` (after observables are imported),
proportional to province area / observation count, normalised so Σw = N,
clipped to [w_min, w_max], then stored as `ref.df['weight']`.


In [ ]:
force_read_weights = False  # set True to recompute from scratch

if "weight" in ref.df.columns and not force_read_weights:
    print("Weights already present – skipping recalculation")
else:
    ref.df["weight"] = calculate_weights(ref.df)
    ref.df.to_parquet(parquet_ref)
    print(f"Weights written, parquet updated: {parquet_ref}")

ref.df["weight"].describe()


In [ ]:
ref.df.to_parquet(parquet_ref)
print(f"Reference parquet saved: {parquet_ref}  {len(ref.df):,} rows")

In [ ]:
for d in dd:
    print(f"{d['label']}: {d['import_type']} (depends on: {d.get('depends_on', [])})")

## 6. KDE diagnostic plot – cleaning pipeline

Shows how the heat-flow distribution changes at each filtering step.
The final **dashed black** curve is the tectonic-province-**weighted** KDE —
the effective training distribution seen by the models.

Legend entries include **n**, **mean**, **median** for each step,
using the seaborn colorblind palette (same colours as original).

Figure saved to `fig/IHFC_q_distribution_cleaning_steps.png`.
Distribution statistics saved to `data/IHFC_distribution_stats.csv`.


In [ ]:
if not steps:
    print("steps list is empty (data loaded from parquet cache).\n"
          "Re-run §3 with force_read=True to regenerate KDE.")
else:
    xmin, xmax = q_min, q_max
    x = np.linspace(xmin, xmax, 512)
    milli_scale = 1000.0

    plt.style.use("seaborn-v0_8-white")
    fig, ax = plt.subplots(figsize=(8, 4), dpi=300)

    colors   = sns.color_palette("colorblind", n_colors=len(steps))
    bw_scale = 0.4

    mean_x,   mean_y   = [], []
    median_x, median_y = [], []
    legend_handles, legend_labels = [], []

    # ── Unweighted KDEs (one per cleaning step) ──────────────────────────
    for step, color in zip(steps, colors):
        q = np.asarray(step["q"])
        n = step["n"]
        if n < 2:
            continue

        kde, y_counts, n_eff = compute_kde(x, q, bw_scale=bw_scale, weights=None)
        ax.plot(x, y_counts, color=color, lw=1.5)

        m_mean   = step["mean"]
        m_median = step["median"]
        y_mean_v   = kde(np.array([m_mean]))[0]   * n_eff
        y_median_v = kde(np.array([m_median]))[0] * n_eff

        ax.plot(m_mean,   y_mean_v,   marker=".", color=color, markersize=7)
        ax.plot(m_median, y_median_v, marker=".", color=color, markersize=7)

        mean_x.append(m_mean);     mean_y.append(y_mean_v)
        median_x.append(m_median); median_y.append(y_median_v)

        handle = plt.Line2D(
            [0], [0], marker="s", color="none",
            markerfacecolor=color, markeredgecolor=color,
            markersize=8, linestyle="None",
        )
        legend_handles.append(handle)
        legend_labels.append(
            f"{step['label']:<26s}  "
            f"N={n:6,d}  "
            f"mean={m_mean * milli_scale:7.1f}  "
            f"med={m_median * milli_scale:7.1f}"
        )

    # ── Weighted KDE (final, using ref.df['weight']) ─────────────────────
    q_last = ref.df["q"].to_numpy()
    w_last = ref.df["weight"].to_numpy()

    if len(q_last) >= 2:
        color_w = "k"
        kde_w, y_counts_w, n_last = compute_kde(
            x, q_last, bw_scale=bw_scale, weights=w_last
        )
        ax.plot(x, y_counts_w, color=color_w, lw=1.5, linestyle="--")

        m_mean_w   = np.average(q_last, weights=w_last)
        m_median_w = weighted_quantile(q_last, 0.5, sample_weight=w_last)
        y_mean_w   = kde_w(np.array([m_mean_w]))[0]   * n_last
        y_median_w = kde_w(np.array([m_median_w]))[0] * n_last

        ax.plot(m_mean_w,   y_mean_w,   marker=".", color=color_w, markersize=7)
        ax.plot(m_median_w, y_median_w, marker=".", color=color_w, markersize=7)

        mean_x.append(m_mean_w);     mean_y.append(y_mean_w)
        median_x.append(m_median_w); median_y.append(y_median_w)

        handle_w = plt.Line2D(
            [0], [0], marker="s", color="none",
            markerfacecolor=color_w, markeredgecolor=color_w,
            markersize=8, linestyle="None",
        )
        legend_handles.append(handle_w)
        legend_labels.append(
            f"{'Weighted by crustal type':<26s}  "
            f"Sum={n_last:6,.1f}  "
            f"mean={m_mean_w * milli_scale:7.1f}  "
            f"med={m_median_w * milli_scale:7.1f}"
        )

        legend_labels.append(
            f"{'Weighted by crustal type':<26s}  "
            f"N={n_last:6,.1f}  "
            f"mean={m_mean_w * milli_scale:7.1f}  "
            f"med={m_median_w * milli_scale:7.1f}"
        )

    # ── Mean / median connector lines ────────────────────────────────────
    if len(mean_x) >= 2:
        ax.plot(mean_x,   mean_y,   linestyle="--", color="gray", lw=1.8,
                label="Mean",   zorder=1)
    if len(median_x) >= 2:
        ax.plot(median_x, median_y, linestyle="--", color="gray", lw=1.8,
                label="Median", zorder=1)

    dy = 0.02 * max(max(mean_y), max(median_y))
    ax.text(mean_x[0],   mean_y[0]   + dy, "Mean q",   ha="left", va="bottom", color="0.3")
    ax.text(median_x[0], median_y[0] + dy, "Median q", ha="left", va="bottom", color="0.3")

    # ── Axes ─────────────────────────────────────────────────────────────
    def _w_to_mw(value, pos): return f"{value * 1000:.0f}"

    ax.set_xlim(xmin, xmax)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(_w_to_mw))
    ax.set_xlabel("Heat flow q  [mW m$^{-2}$]")
    ax.set_ylabel("")
    ax.yaxis.set_ticks([])
    ax.spines["left"].set_visible(False)
    sns.despine(ax=ax, top=True, right=True, left=True, bottom=False, trim=True)

    leg = ax.legend(
        legend_handles, legend_labels,
        loc="upper right", frameon=False,
        prop={"family": "monospace", "size": 7},
    )
    fig.tight_layout(pad=0.0)
    # FIX: use fig_root path (config variable) rather than a bare name
    fig.savefig(fig_root / "IHFC_q_distribution_cleaning_steps.png",
                dpi=300, bbox_inches=None, transparent=True)
    plt.show()
    print(f"Saved {fig_root / 'IHFC_q_distribution_cleaning_steps.png'}")

# ── Distribution statistics CSV ──────────────────────────────────────────
rows = []
for step in (steps if steps else []):
    q = np.asarray(step["q"])
    if len(q) < 2:
        continue
    rows.append({
        "label":      step["label"],
        "n":          step["n"],
        "mean_Wm2":   step["mean"],
        "median_Wm2": step["median"],
        "mean_mWm2":  step["mean"]   * 1000,
        "median_mWm2":step["median"] * 1000,
        "std_Wm2":    float(np.std(q)),
        "p5_Wm2":     float(np.percentile(q, 5)),
        "p95_Wm2":    float(np.percentile(q, 95)),
    })
if "weight" in ref.df.columns and len(ref.df) >= 2:
    q_w = ref.df["q"].to_numpy(); w_w = ref.df["weight"].to_numpy()
    rows.append({
        "label":      "Weighted by crustal type",
        "n":          float(w_w.sum()),
        "mean_Wm2":   float(np.average(q_w, weights=w_w)),
        "median_Wm2": float(weighted_quantile(q_w, 0.5, sample_weight=w_w)),
        "mean_mWm2":  float(np.average(q_w, weights=w_w) * 1000),
        "median_mWm2":float(weighted_quantile(q_w, 0.5, sample_weight=w_w) * 1000),
        "std_Wm2":    float(np.sqrt(np.average(
            (q_w - np.average(q_w, weights=w_w))**2, weights=w_w))),
        "p5_Wm2":     float(weighted_quantile(q_w, 0.05, sample_weight=w_w)),
        "p95_Wm2":    float(weighted_quantile(q_w, 0.95, sample_weight=w_w)),
    })
if rows:
    out_csv = Path("data/IHFC_distribution_stats.csv")
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"Saved distribution stats → {out_csv}  ({len(rows)} rows)")


## 7. Build Antarctica 5 km prediction grid

Regular 5 km grid in EPSG:3031, subsampled from BedMachine Antarctica
(500 m native resolution, step=10 for 5 km), reprojected to WGS84.


In [ ]:
force_read_ant = True

if parquet_ant.exists() and not force_read_ant:
    print(f"Loading cached Antarctica grid from {parquet_ant}")
    ant_df = pd.read_parquet(parquet_ant)
    _ny = _nx = int(round(len(ant_df) ** 0.5))
    ant = Grid(
        lats=ant_df.lat.values, lons=ant_df.lon.values,
        x=ant_df.x.values, y=ant_df.y.values,
        verbose=False, regular_grid=(_ny, _nx),
        name="Antarctica", crs=3031,
        log_file=str(log_dir / "read_ant.log"),
    )
    for col in ant_df.columns:
        ant.df[col] = ant_df[col].values
    print(f"Loaded {len(ant.df):,} points, cols: {list(ant.df.columns)}")
else:
    with xr.open_dataset(bedmachine_antarctica_file) as bm:
        bm_x = bm['x'].values.copy()
        bm_y = bm['y'].values.copy()

    bm_res_m = 500
    step = grid_spacing_m_ant // bm_res_m
    ant_xs = bm_x[::step]
    ant_ys = bm_y[::step]
    XX, YY = np.meshgrid(ant_xs, ant_ys)
    ant_x = XX.ravel()
    ant_y = YY.ravel()
    ny_ant, nx_ant = XX.shape

    transformer = Transformer.from_crs(ant_crs, ref_crs, always_xy=True)
    ant_lon, ant_lat = transformer.transform(ant_x, ant_y)
    print(f'Antarctica grid: {len(ant_x):,} points  ({ny_ant} × {nx_ant})')

    ant = Grid(
        lats=ant_lat, lons=ant_lon, x=ant_x, y=ant_y,
        verbose=False, regular_grid=(ny_ant, nx_ant),
        name='Antarctica', crs=3031,
        log_file=str(log_dir / 'read_ant.log'),
    )
    dd_ant = ant.organise_recipe(recipe.dd)
    imported_ant = ant.read_recipe(dd_ant, verbose=True)

    print(f'\n✓ Antarctica: {len(imported_ant)} observables imported')
    print(f'  Columns: {list(ant.df.columns)}')

    ant.df.to_parquet(parquet_ant)
    print(f'  Cached to {parquet_ant}')


## 8. Build Greenland 5 km prediction grid

Regular 5 km grid in EPSG:3413, subsampled from BedMachine Greenland
(150 m native resolution, step=33 for ~5 km), reprojected to WGS84.


In [ ]:
with xr.open_dataset(bedmachine_greenland_file) as bm:
    bm_x = bm['x'].values.copy()
    bm_y = bm['y'].values.copy()

bm_res_m = 150
step = grid_spacing_m_grl // bm_res_m   # = 33 for 5 km

grl_xs = bm_x[::step]
grl_ys = bm_y[::step]
XX, YY = np.meshgrid(grl_xs, grl_ys)
grl_x = XX.ravel()
grl_y = YY.ravel()
ny_grl, nx_grl = XX.shape

transformer_grl = Transformer.from_crs(grl_crs, ref_crs, always_xy=True)
grl_lon, grl_lat = transformer_grl.transform(grl_x, grl_y)
print(f'Greenland grid: {len(grl_x):,} points  ({ny_grl} × {nx_grl})')

grl = Grid(
    lats=grl_lat,
    lons=grl_lon,
    x=grl_x,
    y=grl_y,
    name="Greenland",  
    verbose=False,
    regular_grid=(ny_grl, nx_grl),
    crs=3413,
    log_file=str(log_dir / 'read_grl.log'),
)

dd_grl = grl.organise_recipe(recipe.dd)
imported_grl = grl.read_recipe(dd_grl, verbose=True)
print(f'\n✓ Greenland: {len(imported_grl)} observables imported')
print(f'  Columns: {list(grl.df.columns)}')

grl.df.to_parquet(parquet_grl)
print(f'  Cached to {parquet_grl}')


## 9. Summary validation

In [ ]:
for name, grid in [("ref", ref), ("ant", ant), ("grl", grl)]:
    print(f"--- {name} ---")
    print(f"  Points:  {len(grid.df):,}")
    print(f"  Columns: {list(grid.df.columns)}")
    if "DEM" in grid.df.columns:
        print(f"  DEM range: {grid.df.DEM.min():.0f} – {grid.df.DEM.max():.0f} m")
    nan_frac = grid.df.isna().mean()
    high_nan = nan_frac[nan_frac > 0.05].to_dict()
    if high_nan:
        print(f"  High NaN (>5%): {high_nan}")
    print()


## 10. Quick-look: reference heat-flow map (`q`)

In [ ]:
# display-only; refined plotting moves to notebook 2_Observables
ref.map(
    "q",
    vmin=q_min, vmax=q_max,
    cmap=hf_cmap,
    gridlines={"step": 30},
    no_frame=False,
    savefig=str(fig_root / "IHFC_q_map.png"),
)


## 11. Quick-look: all observables – ref, Antarctica, Greenland

Uses `Grid.quicklook()`.  Attempts 2D grid rendering; falls back to scatter
for irregular data.  One figure per grid, all recipe observables present in
that grid's DataFrame.


In [ ]:
obs_labels_ref = [
    d.get("recipe_id", d["label"]) for d in recipe.dd
    if d.get("recipe_id", d["label"]) in ref.df.columns
]
print(f"Quick-look ref ({len(obs_labels_ref)} observables)")
ref.quicklook(obs_labels_ref, recipe.dd, show=True);


In [ ]:
obs_labels_ant = [
    d.get("recipe_id", d["label"]) for d in recipe.dd
    if d.get("recipe_id", d["label"]) in ant.df.columns
]
print(f"Quick-look Antarctica ({len(obs_labels_ant)} observables)")
ant.quicklook(obs_labels_ant, recipe.dd, show=True);


In [ ]:
obs_labels_grl = [
    d.get("recipe_id", d["label"]) for d in recipe.dd
    if d.get("recipe_id", d["label"]) in grl.df.columns
]
print(f"Quick-look Greenland ({len(obs_labels_grl)} observables)")
grl.quicklook(obs_labels_grl, recipe.dd, show=True, ncols=6);


In [ ]:
def observable_stats(grid, recipe_dd):
    recipe_map = {d["label"]: d for d in recipe_dd}

    skip = {"lon", "lat", "x", "y", "elevation", "Domain",
            "QualityCode", "cluster", "weight"}
    obs_cols = [c for c in grid.df.columns if c not in skip]

    rows = []
    for label in obs_cols:
        # Skip non-numeric columns
        if not pd.api.types.is_numeric_dtype(grid.df[label]):
            continue

        arr = grid.df[label].to_numpy(dtype=float)
        valid = arr[~np.isnan(arr)]
        n_total = len(arr)
        n_valid = len(valid)

        rd = recipe_map.get(label, {})
        vr = rd.get("v_range", None)
        grid_key = rd.get("grid", None)

        row = dict(
            label       = label,
            grid        = grid_key if grid_key else "shared",
            n_total     = n_total,
            n_valid     = n_valid,
            nan_ratio   = round((n_total - n_valid) / n_total, 6) if n_total else np.nan,
            vmin_recipe = vr[0] if vr is not None else np.nan,
            vmax_recipe = vr[1] if vr is not None else np.nan,
            p01         = round(float(np.nanpercentile(valid,  1)), 4) if n_valid else np.nan,
            p99         = round(float(np.nanpercentile(valid, 99)), 4) if n_valid else np.nan,
            median      = round(float(np.nanmedian(valid)),         4) if n_valid else np.nan,
            mean        = round(float(np.nanmean(valid)),           4) if n_valid else np.nan,
            std         = round(float(np.nanstd(valid)),            4) if n_valid else np.nan,
            data_min    = round(float(np.nanmin(valid)),              4) if n_valid else np.nan,
            data_max    = round(float(np.nanmax(valid)),              4) if n_valid else np.nan,
        )
        rows.append(row)

    return pd.DataFrame(rows)

# ── Run for each grid and save ──────────────────────────────────────────────
for name, grid in [("ref", ref), ("antarctica", ant), ("greenland", grl)]:
    df_stats = observable_stats(grid, recipe.dd)
    out_path = local_data / f"observables_{name}.csv"
    df_stats.to_csv(out_path, index=False)
    print(f"Saved {out_path}  ({len(df_stats)} observables)")
    display(df_stats)